# PART 1 — PySpark Fundamentals & Environment
### A complete beginner-to-expert PySpark learning 


## 1. What is Spark? Spark Architecture

### What is Apache Spark?
Apache Spark is a **distributed, in-memory data processing engine**. Instead of processing
a huge file on ONE machine (slow, may not even fit in memory), Spark **splits the data into
chunks (partitions) and processes those chunks in parallel across many machines** (a cluster).

**Real-life analogy:**
Imagine you must count all books in a 10-floor library by yourself — it would take hours.
Now imagine you hire 10 people, one per floor, all counting **at the same time**, and then
you add up their totals. That's exactly what Spark does with data: it divides the work
(partitions) and does it in parallel (distributed computing).

### Spark Architecture — 3 Main Components

```
                        ┌────────────────────────────┐
                        │        CLUSTER MANAGER      │
                        │  (YARN / Kubernetes / Databricks) │
                        │  Decides which machines      │
                        │  (nodes) are available        │
                        └───────────────┬──────────────┘
                                         │ allocates resources
                                         ▼
  ┌───────────────────────────────────────────────────────────────────┐
  │                              DRIVER                                │
  │  - Runs your main() program / notebook code                        │
  │  - Creates the SparkSession                                        │
  │  - Builds the DAG (Directed Acyclic Graph) of operations           │
  │  - Splits work into JOBS -> STAGES -> TASKS, sends to executors     │
  │  - Collects final results back                                     │
  └───────────────┬───────────────────────┬───────────────┬────────────┘
                  │                       │               │
                  ▼                       ▼               ▼
         ┌─────────────────┐   ┌─────────────────┐  ┌─────────────────┐
         │   EXECUTOR 1    │   │   EXECUTOR 2    │  │   EXECUTOR 3    │
         │  runs TASKS     │   │  runs TASKS     │  │  runs TASKS     │
         │  holds data in  │   │  holds data in  │  │  holds data in  │
         │  memory/disk    │   │  memory/disk    │  │  memory/disk    │
         └─────────────────┘   └─────────────────┘  └─────────────────┘
             (Worker Node 1)       (Worker Node 2)      (Worker Node 3)
```

| Component | Role | Real-life analogy |
|---|---|---|
| **Driver** | The "brain" — plans the work, breaks it into tasks, coordinates everything | Restaurant Manager who takes the order and assigns dishes to chefs |
| **Executors** | Worker processes that actually do the computation and hold data in memory/disk | Chefs, each cooking their assigned dish |
| **Cluster Manager** | Allocates machines/resources (CPU, RAM) to the Driver & Executors | Restaurant Owner who decides how many chefs are hired today |

### Job → Stage → Task hierarchy (how the Driver breaks down work)
```
  ACTION (e.g. .show(), .count())
       │
       ▼
     JOB  (one job is triggered per ACTION)
       │
       ├── STAGE 1  (a group of transformations that don't need a SHUFFLE)
       │      ├── Task 1  (runs on partition 1)
       │      ├── Task 2  (runs on partition 2)
       │      └── Task 3  (runs on partition 3)
       │
       └── STAGE 2  (new stage starts whenever data must be SHUFFLED, e.g. groupBy/join)
              ├── Task 1
              └── Task 2
```
**Key idea — Transformations vs Actions:**
- **Transformations** (select, filter, withColumn...) are **LAZY** — Spark just builds a plan, nothing runs yet.
- **Actions** (show, count, collect, write...) **TRIGGER** actual execution across the cluster.
This laziness lets Spark's Catalyst Optimizer look at your ENTIRE chain of transformations and find the fastest execution plan before running anything — like a GPS calculating the best route before you start driving, instead of turn-by-turn with no overview.

In [0]:
# In Databricks, a SparkSession called `spark` is ALREADY created for you automatically.
# You do NOT need to create it yourself in a Databricks notebook (unlike plain PySpark scripts).

# Let's inspect the Driver and cluster info to see the architecture in action.
print("Spark Version:", spark.version)                       # Shows the Spark engine version running on the Driver
print("App Name:", spark.sparkContext.appName)                # Name of this Spark Application
print("Master:", spark.sparkContext.master)                   # Shows cluster manager connection info
print("Default Parallelism (approx total cores across executors):",
      spark.sparkContext.defaultParallelism)                  # How many tasks can run in parallel = total cores available

In [0]:
# ---- See Jobs/Stages/Tasks live: run a transformation chain then trigger an action ----
sample_df = spark.range(1, 1000001)                    # LAZY: just builds a plan for numbers 1 to 1,000,000 (nothing executes yet)
sample_df2 = sample_df.withColumn("double_val", sample_df.id * 2)   # LAZY: still just a plan
sample_df3 = sample_df2.filter(sample_df2.double_val > 500000)       # LAZY: still just a plan

print("Nothing has run yet - transformations are lazy!")

result_count = sample_df3.count()                       # ACTION: THIS triggers a real JOB on the cluster
print("Count of rows where double_val > 500000:", result_count)
# TIP: Open the "Spark UI" tab (top of notebook results) -> Jobs -> you'll see this single .count() created 1 Job with Stages/Tasks

In [0]:
sample_df3.show()

In [0]:
print(sample_df3)

In [0]:
sample_df3.display()

In [0]:
display(sample_df3.limit(100))